# Advanced Python Context Managers — Problems with Complete Solutions

This notebook develops production-quality context managers around the three core patterns:

1. **Open → Close**
2. **Change → Reset**
3. **Start → Stop**

It goes beyond introductory examples and emphasizes exception safety, re-entrancy, composability, dynamic resource management, concurrency, asynchronous cleanup, and testing.

## Learning objectives

By the end, you will be able to:

- implement class-based and generator-based context managers;
- decide what `__enter__` should return;
- write correct `__exit__` exception semantics;
- restore state even when the body fails;
- support nested and re-entrant use safely;
- manage a dynamic number of resources with `ExitStack` and `AsyncExitStack`;
- use context managers as decorators;
- design transactional rollback behavior;
- test cleanup guarantees rather than only happy-path results.

> All solutions use only the Python standard library.

## Best-practices checklist

- Acquire or snapshot mutable state in `__enter__`, not usually in `__init__`.
- Release resources in `__exit__` even when exceptions occur.
- Return `False` or `None` from `__exit__` unless suppression is deliberate.
- Suppress only narrow, explicitly documented exception types.
- Make cleanup idempotent when practical.
- Prefer built-ins from `contextlib` when they already express the behavior.
- Use a stack, counter, or context-local state for nested/re-entrant use.
- Test normal completion, exceptional completion, nesting, and partial acquisition failure.
- Avoid returning from a `finally` block because it can hide exceptions.
- Keep the context manager focused on lifecycle/state boundaries.

## Shared setup

In [1]:
from __future__ import annotations

import asyncio
import contextlib
import contextvars
import decimal
import html
import io
import os
import tempfile
import threading
import time
from collections import defaultdict
from dataclasses import dataclass, field
from decimal import Decimal
from pathlib import Path
from time import perf_counter
from typing import Any, AsyncIterator, Callable, Iterable, Iterator, Mapping, MutableMapping

def decimal_context_signature(ctx: decimal.Context) -> tuple[Any, ...]:
    """Return a structural snapshot suitable for restoration tests."""
    return (
        ctx.prec,
        ctx.rounding,
        ctx.Emin,
        ctx.Emax,
        ctx.capitals,
        ctx.clamp,
        tuple(ctx.flags.items()),
        tuple(ctx.traps.items()),
    )


print("Python context-manager lab ready")

Python context-manager lab ready


---
## Problem 1 — Exception-safe decimal precision

Implement a class-based context manager named `decimal_precision`.

### Requirements

- Temporarily change decimal precision.
- Capture the previous state when entering, not when constructing the object.
- Restore the exact previous context after normal completion or an exception.
- Return the active decimal context from `__enter__`.
- Validate that precision is a positive integer.

Why capture state in `__enter__`? An instance may be created long before it is used; global state may change in between.

### Solution

In [2]:
class decimal_precision:
    """Temporarily replace the active Decimal context precision."""

    def __init__(self, precision: int):
        if isinstance(precision, bool) or not isinstance(precision, int):
            raise TypeError("precision must be an integer")
        if precision <= 0:
            raise ValueError("precision must be positive")
        self.precision = precision
        self._manager: contextlib.AbstractContextManager | None = None

    def __enter__(self) -> decimal.Context:
        # decimal.localcontext() copies the active context on entry and restores it on exit.
        self._manager = decimal.localcontext()
        active = self._manager.__enter__()
        active.prec = self.precision
        return active

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self._manager is None:
            raise RuntimeError("context was never entered")
        return bool(self._manager.__exit__(exc_type, exc_value, traceback))

In [3]:
original = decimal.getcontext().copy()

with decimal_precision(6) as ctx:
    assert ctx.prec == 6
    result = Decimal(1) / Decimal(7)
    print("inside:", result)

assert decimal_context_signature(decimal.getcontext()) == decimal_context_signature(original)
print("outside precision:", decimal.getcontext().prec)

try:
    with decimal_precision(4):
        assert str(Decimal(1) / Decimal(3)) == "0.3333"
        raise RuntimeError("simulated failure")
except RuntimeError:
    pass

assert decimal_context_signature(decimal.getcontext()) == decimal_context_signature(original)
print("restoration after exception: OK")

inside: 0.142857
outside precision: 28
restoration after exception: OK


---
## Problem 2 — A configurable decimal context with traps and rounding

Create a generator-based context manager `decimal_settings` that can temporarily override:

- precision;
- rounding mode;
- selected decimal traps.

The caller should be able to write:

```python
with decimal_settings(precision=8, rounding=decimal.ROUND_HALF_UP,
                      traps={decimal.DivisionByZero: False}) as ctx:
    ...
```

Do not mutate the caller's original context permanently.

### Solution

In [4]:
@contextlib.contextmanager
def decimal_settings(
    *,
    precision: int | None = None,
    rounding: str | None = None,
    traps: Mapping[type[BaseException], bool] | None = None,
) -> Iterator[decimal.Context]:
    with decimal.localcontext() as ctx:
        if precision is not None:
            if isinstance(precision, bool) or not isinstance(precision, int) or precision <= 0:
                raise ValueError("precision must be a positive integer")
            ctx.prec = precision

        if rounding is not None:
            valid_roundings = {
                decimal.ROUND_CEILING,
                decimal.ROUND_DOWN,
                decimal.ROUND_FLOOR,
                decimal.ROUND_HALF_DOWN,
                decimal.ROUND_HALF_EVEN,
                decimal.ROUND_HALF_UP,
                decimal.ROUND_UP,
                decimal.ROUND_05UP,
            }
            if rounding not in valid_roundings:
                raise ValueError(f"unsupported rounding mode: {rounding!r}")
            ctx.rounding = rounding

        if traps:
            for signal, enabled in traps.items():
                if signal not in ctx.traps:
                    raise KeyError(f"unknown decimal signal: {signal}")
                ctx.traps[signal] = bool(enabled)

        yield ctx

In [5]:
outer = decimal.getcontext().copy()

with decimal_settings(
    precision=8,
    rounding=decimal.ROUND_HALF_UP,
    traps={decimal.DivisionByZero: False},
) as ctx:
    rounded = Decimal("2.345").quantize(Decimal("0.01"))
    divided = Decimal(1) / Decimal(0)
    print("rounded:", rounded)
    print("division result:", divided)
    assert rounded == Decimal("2.35")
    assert divided.is_infinite()

assert decimal_context_signature(decimal.getcontext()) == decimal_context_signature(outer)
print("multi-setting restoration: OK")

rounded: 2.35
division result: Infinity
multi-setting restoration: OK


---
## Problem 3 — A reusable timer with laps and failure metadata

Build a timer that:

- measures elapsed time with `perf_counter`;
- returns itself from `__enter__`;
- records named laps;
- records whether the block failed and the exception type;
- can be reused sequentially;
- rejects overlapping/re-entrant entry of the same instance.

### Solution

In [6]:
@dataclass
class Timer:
    elapsed: float = 0.0
    laps: list[tuple[str, float]] = field(default_factory=list)
    failed: bool = False
    exception_type: type[BaseException] | None = None
    _started_at: float | None = field(default=None, init=False, repr=False)
    _last_lap_at: float | None = field(default=None, init=False, repr=False)

    def __enter__(self) -> "Timer":
        if self._started_at is not None:
            raise RuntimeError("the same Timer instance cannot overlap itself")
        now = perf_counter()
        self.elapsed = 0.0
        self.laps.clear()
        self.failed = False
        self.exception_type = None
        self._started_at = self._last_lap_at = now
        return self

    def lap(self, name: str) -> float:
        if self._last_lap_at is None:
            raise RuntimeError("timer is not active")
        now = perf_counter()
        duration = now - self._last_lap_at
        self._last_lap_at = now
        self.laps.append((name, duration))
        return duration

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self._started_at is None:
            raise RuntimeError("timer is not active")
        self.elapsed = perf_counter() - self._started_at
        self.failed = exc_type is not None
        self.exception_type = exc_type
        self._started_at = None
        self._last_lap_at = None
        return False

In [7]:
timer = Timer()

with timer:
    time.sleep(0.01)
    timer.lap("phase A")
    time.sleep(0.005)
    timer.lap("phase B")

assert timer.elapsed >= 0.015
assert [name for name, _ in timer.laps] == ["phase A", "phase B"]
assert not timer.failed
print(f"elapsed={timer.elapsed:.4f}s, laps={timer.laps}")

try:
    with timer:
        raise LookupError("boom")
except LookupError:
    pass

assert timer.failed and timer.exception_type is LookupError
print("failure metadata:", timer.exception_type.__name__)

elapsed=0.0164s, laps=[('phase A', 0.010491000022739172), ('phase B', 0.005882099736481905)]
failure metadata: LookupError


---
## Problem 4 — Capture both stdout and stderr

Write `capture_output()` as a generator-based context manager.

It must:

- capture `stdout` and `stderr` independently;
- yield an object whose `.stdout` and `.stderr` buffers can be inspected;
- restore the original streams after exceptions;
- compose existing `contextlib` utilities rather than assigning to `sys.stdout` manually.

### Solution

In [8]:
@dataclass
class CapturedOutput:
    stdout: io.StringIO
    stderr: io.StringIO


@contextlib.contextmanager
def capture_output() -> Iterator[CapturedOutput]:
    out = io.StringIO()
    err = io.StringIO()
    captured = CapturedOutput(stdout=out, stderr=err)
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):
        yield captured

In [9]:
import sys

with capture_output() as captured:
    print("normal message")
    print("warning message", file=sys.stderr)

assert captured.stdout.getvalue() == "normal message\n"
assert captured.stderr.getvalue() == "warning message\n"
print("stdout:", captured.stdout.getvalue().strip())
print("stderr:", captured.stderr.getvalue().strip())

try:
    with capture_output() as failed_capture:
        print("before failure")
        raise ValueError("expected")
except ValueError:
    pass

assert failed_capture.stdout.getvalue() == "before failure\n"
print("stream restoration after exception: OK")

stdout: normal message
stderr: warning message
stream restoration after exception: OK


---
## Problem 5 — Safe nested HTML generation

A context manager that prints raw tags can accidentally emit invalid or unsafe markup. Build `HTMLWriter` with:

- escaped text nodes;
- validated tag names;
- escaped attribute values;
- nested tags through a context manager method;
- deterministic indentation;
- automatic closing tags after exceptions.

The writer should accumulate HTML in memory rather than changing global output.

### Solution

In [10]:
class HTMLWriter:
    def __init__(self, indent: str = "  "):
        self._indent_unit = indent
        self._depth = 0
        self._lines: list[str] = []

    @staticmethod
    def _validate_tag(tag: str) -> str:
        if not tag or not tag.replace("-", "").isalnum() or not tag[0].isalpha():
            raise ValueError(f"invalid tag name: {tag!r}")
        return tag

    def _prefix(self) -> str:
        return self._indent_unit * self._depth

    def text(self, value: Any) -> None:
        self._lines.append(self._prefix() + html.escape(str(value), quote=False))

    @contextlib.contextmanager
    def tag(self, tag: str, **attributes: Any) -> Iterator["HTMLWriter"]:
        tag = self._validate_tag(tag)
        rendered_attrs = "".join(
            f' {name.rstrip("_").replace("_", "-")}="{html.escape(str(value), quote=True)}"'
            for name, value in attributes.items()
        )
        self._lines.append(f"{self._prefix()}<{tag}{rendered_attrs}>")
        self._depth += 1
        try:
            yield self
        finally:
            self._depth -= 1
            self._lines.append(f"{self._prefix()}</{tag}>")

    def render(self) -> str:
        if self._depth != 0:
            raise RuntimeError("cannot render while a tag context is still open")
        return "\n".join(self._lines)

In [11]:
writer = HTMLWriter()
with writer.tag("article", class_="lesson", data_id='7" onclick="bad'):
    with writer.tag("h1"):
        writer.text("Context Managers <Advanced>")
    with writer.tag("p"):
        writer.text("Use <, >, and & safely.")

rendered = writer.render()
print(rendered)
assert "&lt;Advanced&gt;" in rendered
assert "&amp;" in rendered
assert "&quot;" in rendered
assert rendered.count("<article") == 1 and rendered.count("</article>") == 1

<article class="lesson" data-id="7&quot; onclick=&quot;bad">
  <h1>
    Context Managers &lt;Advanced&gt;
  </h1>
  <p>
    Use &lt;, &gt;, and &amp; safely.
  </p>
</article>


---
## Problem 6 — Re-entrant indentation with context-local state

A simple indentation counter stored on one object can break when the object is shared across asynchronous tasks or threads. Implement `IndentWriter` using `contextvars.ContextVar`.

### Requirements

- The same writer must be nestable and re-entrant.
- Indentation state must be local to the current execution context.
- `level()` should return a context manager.
- Incorrect exit ordering should be impossible because `ContextVar.reset(token)` restores the exact previous value.

### Solution

In [12]:
class IndentWriter:
    def __init__(self, prefix: str = "- ", indent: str = "   "):
        self.prefix = prefix
        self.indent = indent
        self._depth = contextvars.ContextVar(f"indent_depth_{id(self)}", default=0)
        self.lines: list[str] = []

    def write(self, value: Any) -> None:
        depth = self._depth.get()
        self.lines.append(self.indent * depth + self.prefix + str(value))

    @contextlib.contextmanager
    def level(self) -> Iterator["IndentWriter"]:
        token = self._depth.set(self._depth.get() + 1)
        try:
            yield self
        finally:
            self._depth.reset(token)

    def render(self) -> str:
        return "\n".join(self.lines)

In [13]:
outline = IndentWriter()
outline.write("root")
with outline.level():
    outline.write("child 1")
    with outline.level():
        outline.write("grandchild")
    outline.write("child 2")
outline.write("root 2")

print(outline.render())
assert outline.lines == [
    "- root",
    "   - child 1",
    "      - grandchild",
    "   - child 2",
    "- root 2",
]

- root
   - child 1
      - grandchild
   - child 2
- root 2


---
## Problem 7 — Selective exception suppression with an audit record

Implement `suppress_and_record(*exception_types)`.

It should:

- suppress only the declared exception classes;
- store the caught exception object;
- expose whether suppression happened;
- never suppress `BaseException` subclasses such as `KeyboardInterrupt` unless the caller explicitly asks for them;
- reject an empty exception list.

### Solution

In [14]:
class suppress_and_record:
    def __init__(self, *exception_types: type[BaseException]):
        if not exception_types:
            raise ValueError("at least one exception type is required")
        if not all(isinstance(t, type) and issubclass(t, BaseException) for t in exception_types):
            raise TypeError("all arguments must be exception classes")
        self.exception_types = exception_types
        self.exception: BaseException | None = None
        self.suppressed = False

    def __enter__(self) -> "suppress_and_record":
        self.exception = None
        self.suppressed = False
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type is None:
            return False
        if issubclass(exc_type, self.exception_types):
            self.exception = exc_value
            self.suppressed = True
            return True
        return False

In [15]:
with suppress_and_record(KeyError, IndexError) as guard:
    {}["missing"]

assert guard.suppressed
assert isinstance(guard.exception, KeyError)
print("suppressed:", type(guard.exception).__name__)

try:
    with suppress_and_record(KeyError):
        raise ValueError("must propagate")
except ValueError as exc:
    print("propagated:", type(exc).__name__)
else:
    raise AssertionError("ValueError should not have been suppressed")

suppressed: KeyError
propagated: ValueError


---
## Problem 8 — Transactional mapping with rollback

Create `mapping_transaction(mapping)`.

Behavior:

- On success, keep all changes.
- On failure, restore the exact original contents.
- Preserve the identity of the original mapping object.
- Support nested transactions.
- Use a snapshot appropriate for ordinary JSON-like values.

Discuss the limitation: a shallow snapshot cannot undo in-place mutation of nested mutable objects, so this solution uses `copy.deepcopy`.

### Solution

In [16]:
import copy


@contextlib.contextmanager
def mapping_transaction(mapping: MutableMapping[Any, Any]) -> Iterator[MutableMapping[Any, Any]]:
    snapshot = copy.deepcopy(dict(mapping))
    try:
        yield mapping
    except BaseException:
        mapping.clear()
        mapping.update(snapshot)
        raise

In [17]:
state = {"balance": 100, "tags": ["new"]}
identity = id(state)

with mapping_transaction(state):
    state["balance"] -= 25
    state["committed"] = True

assert state == {"balance": 75, "tags": ["new"], "committed": True}
assert id(state) == identity

try:
    with mapping_transaction(state):
        state["balance"] = -999
        state["tags"].append("corrupted")
        del state["committed"]
        raise RuntimeError("rollback")
except RuntimeError:
    pass

assert state == {"balance": 75, "tags": ["new"], "committed": True}
assert id(state) == identity
print("rolled back state:", state)

# Nested transaction: the inner rollback does not erase valid outer work.
with mapping_transaction(state):
    state["outer"] = 1
    try:
        with mapping_transaction(state):
            state["inner"] = 2
            raise ValueError
    except ValueError:
        pass
    assert "inner" not in state

assert state["outer"] == 1
print("nested transaction: OK")

rolled back state: {'balance': 75, 'tags': ['new'], 'committed': True}
nested transaction: OK


---
## Problem 9 — Temporary environment variables

Implement `temporary_environ(**changes)`.

Rules:

- Values are converted to strings.
- Passing `None` temporarily removes a variable.
- On exit, restore both values and non-existence exactly.
- Nested use must work.
- Only variables named by the caller should be touched.

### Solution

In [18]:
_MISSING = object()


@contextlib.contextmanager
def temporary_environ(**changes: Any) -> Iterator[None]:
    previous = {name: os.environ.get(name, _MISSING) for name in changes}
    try:
        for name, value in changes.items():
            if value is None:
                os.environ.pop(name, None)
            else:
                os.environ[name] = str(value)
        yield
    finally:
        for name, old_value in previous.items():
            if old_value is _MISSING:
                os.environ.pop(name, None)
            else:
                os.environ[name] = str(old_value)

In [19]:
os.environ["CM_EXISTING"] = "outer"
os.environ.pop("CM_NEW", None)

with temporary_environ(CM_EXISTING="inner", CM_NEW=123):
    assert os.environ["CM_EXISTING"] == "inner"
    assert os.environ["CM_NEW"] == "123"
    with temporary_environ(CM_EXISTING=None):
        assert "CM_EXISTING" not in os.environ
    assert os.environ["CM_EXISTING"] == "inner"

assert os.environ["CM_EXISTING"] == "outer"
assert "CM_NEW" not in os.environ
print("environment restoration: OK")
os.environ.pop("CM_EXISTING", None)

environment restoration: OK


'outer'

---
## Problem 10 — Temporary workspace and current-directory restoration

Create `temporary_workspace()` that:

- creates a temporary directory;
- changes the current working directory to it;
- yields the directory as a `Path`;
- restores the previous working directory before deleting the temporary directory;
- remains safe when the block raises.

Use composition rather than duplicating cleanup logic.

### Solution

In [20]:
@contextlib.contextmanager
def temporary_workspace(prefix: str = "cm-lab-") -> Iterator[Path]:
    previous_cwd = Path.cwd()
    with tempfile.TemporaryDirectory(prefix=prefix) as directory:
        workspace = Path(directory)
        os.chdir(workspace)
        try:
            yield workspace
        finally:
            # Restore cwd before TemporaryDirectory attempts deletion.
            os.chdir(previous_cwd)

In [21]:
start_dir = Path.cwd()
workspace_path: Path | None = None

try:
    with temporary_workspace() as workspace:
        workspace_path = workspace
        Path("data.txt").write_text("context managers", encoding="utf-8")
        assert Path.cwd() == workspace
        assert Path("data.txt").read_text(encoding="utf-8") == "context managers"
        raise RuntimeError("force exceptional cleanup")
except RuntimeError:
    pass

assert Path.cwd() == start_dir
assert workspace_path is not None and not workspace_path.exists()
print("workspace cleanup after exception: OK")

workspace cleanup after exception: OK


---
## Problem 11 — Dynamic resource acquisition with `ExitStack`

Write `open_text_files(paths, mode="r")`.

A normal `with a, b, c:` statement needs a fixed number of resources. Here, the number of paths is known only at runtime.

Requirements:

- Open every file using UTF-8.
- Yield the list of file objects.
- If opening a later file fails, close all files opened earlier.
- Close everything in reverse order.
- Use `ExitStack`.

### Solution

In [22]:
@contextlib.contextmanager
def open_text_files(paths: Iterable[os.PathLike[str] | str], mode: str = "r"):
    with contextlib.ExitStack() as stack:
        files = [
            stack.enter_context(open(path, mode, encoding="utf-8"))
            for path in paths
        ]
        yield files

In [23]:
with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    paths = [root / f"part-{i}.txt" for i in range(3)]

    with open_text_files(paths, "w") as files:
        for index, file in enumerate(files):
            file.write(f"record {index}\n")
        assert all(not file.closed for file in files)

    assert all(file.closed for file in files)

    with open_text_files(paths, "r") as files:
        contents = [file.read().strip() for file in files]

    assert contents == ["record 0", "record 1", "record 2"]
    print(contents)

['record 0', 'record 1', 'record 2']


---
## Problem 12 — Cleanup after partial acquisition failure

`ExitStack` is especially valuable when setup itself can fail. Create a simulated resource and a function `acquire_resources(names, fail_on=None)`.

If acquisition of resource `C` fails, resources `B` and `A` must still close. Verify reverse-order cleanup using an event log.

### Solution

In [24]:
class DemoResource:
    def __init__(self, name: str, events: list[str], fail: bool = False):
        self.name = name
        self.events = events
        self.fail = fail

    def __enter__(self) -> "DemoResource":
        self.events.append(f"enter:{self.name}")
        if self.fail:
            raise RuntimeError(f"failed to acquire {self.name}")
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.events.append(f"exit:{self.name}")
        return False


def acquire_resources(names: Iterable[str], events: list[str], fail_on: str | None = None):
    stack = contextlib.ExitStack()
    try:
        resources = [
            stack.enter_context(DemoResource(name, events, fail=(name == fail_on)))
            for name in names
        ]
    except BaseException:
        stack.close()
        raise
    return stack, resources

In [25]:
events: list[str] = []
try:
    stack, resources = acquire_resources(["A", "B", "C"], events, fail_on="C")
except RuntimeError as exc:
    print(exc)

assert events == ["enter:A", "enter:B", "enter:C", "exit:B", "exit:A"]
print(events)

# Successful acquisition transfers ownership to the returned ExitStack.
events.clear()
stack, resources = acquire_resources(["A", "B"], events)
assert [resource.name for resource in resources] == ["A", "B"]
stack.close()
assert events == ["enter:A", "enter:B", "exit:B", "exit:A"]

failed to acquire C
['enter:A', 'enter:B', 'enter:C', 'exit:B', 'exit:A']


---
## Problem 13 — A context manager that is also a decorator

Create `record_calls`, a `ContextDecorator`, that records timing and failure information for both:

```python
with recorder:
    ...
```

and:

```python
@record_calls(log)
def function(...):
    ...
```

Each use must append a fresh record to a shared list.

### Solution

In [26]:
class record_calls(contextlib.ContextDecorator):
    def __init__(self, log: list[dict[str, Any]], label: str = "operation"):
        self.log = log
        self.label = label
        self._start: float | None = None

    def __enter__(self) -> "record_calls":
        if self._start is not None:
            raise RuntimeError("one recorder instance cannot overlap itself")
        self._start = perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self._start is None:
            raise RuntimeError("recorder was not active")
        self.log.append(
            {
                "label": self.label,
                "elapsed": perf_counter() - self._start,
                "ok": exc_type is None,
                "exception": None if exc_type is None else exc_type.__name__,
            }
        )
        self._start = None
        return False

In [27]:
call_log: list[dict[str, Any]] = []

@record_calls(call_log, "decorated-add")
def add(a: int, b: int) -> int:
    time.sleep(0.002)
    return a + b

assert add(2, 3) == 5

with record_calls(call_log, "with-block"):
    sum(range(10_000))

try:
    with record_calls(call_log, "failing-block"):
        raise ArithmeticError("failure")
except ArithmeticError:
    pass

assert [entry["ok"] for entry in call_log] == [True, True, False]
print(call_log)

[{'label': 'decorated-add', 'elapsed': 0.0024748002178967, 'ok': True, 'exception': None}, {'label': 'with-block', 'elapsed': 0.00039059994742274284, 'ok': True, 'exception': None}, {'label': 'failing-block', 'elapsed': 3.700144588947296e-06, 'ok': False, 'exception': 'ArithmeticError'}]


---
## Problem 14 — Lock acquisition in a global order

Acquiring multiple locks in inconsistent orders can deadlock. Implement `acquire_locks(*locks)` that:

- sorts locks by a stable process-local key (`id`);
- rejects duplicate lock objects;
- acquires all locks in order;
- releases them in reverse order;
- handles acquisition failure with `ExitStack` callbacks.

This pattern reduces deadlock risk only when **all participating code follows the same ordering rule**.

### Solution

In [28]:
@contextlib.contextmanager
def acquire_locks(*locks: threading.Lock) -> Iterator[tuple[threading.Lock, ...]]:
    if len({id(lock) for lock in locks}) != len(locks):
        raise ValueError("duplicate lock objects are not allowed")

    ordered = tuple(sorted(locks, key=id))
    with contextlib.ExitStack() as stack:
        for lock in ordered:
            lock.acquire()
            stack.callback(lock.release)
        yield ordered

In [29]:
lock_a = threading.Lock()
lock_b = threading.Lock()
shared_counter = 0

with acquire_locks(lock_b, lock_a) as ordered:
    assert ordered == tuple(sorted((lock_a, lock_b), key=id))
    shared_counter += 1

assert shared_counter == 1
assert lock_a.acquire(blocking=False)
lock_a.release()
assert lock_b.acquire(blocking=False)
lock_b.release()
print("locks released correctly")

locks released correctly


---
## Problem 15 — An asynchronous resource pool with `AsyncExitStack`

Build a small asynchronous demo with resources that have `__aenter__` and `__aexit__`.

Implement `open_async_resources(names)` so that:

- any runtime number of resources can be opened;
- cleanup occurs in reverse order;
- cleanup still happens if the body raises;
- each resource's lifecycle is recorded.

### Solution

In [30]:
class AsyncDemoResource:
    def __init__(self, name: str, events: list[str]):
        self.name = name
        self.events = events

    async def __aenter__(self) -> "AsyncDemoResource":
        await asyncio.sleep(0)
        self.events.append(f"aenter:{self.name}")
        return self

    async def __aexit__(self, exc_type, exc_value, traceback) -> bool:
        await asyncio.sleep(0)
        self.events.append(f"aexit:{self.name}")
        return False


@contextlib.asynccontextmanager
async def open_async_resources(
    names: Iterable[str], events: list[str]
) -> AsyncIterator[list[AsyncDemoResource]]:
    async with contextlib.AsyncExitStack() as stack:
        resources = [
            await stack.enter_async_context(AsyncDemoResource(name, events))
            for name in names
        ]
        yield resources

In [31]:
async def async_resource_demo() -> list[str]:
    events: list[str] = []
    try:
        async with open_async_resources(["X", "Y", "Z"], events) as resources:
            assert [resource.name for resource in resources] == ["X", "Y", "Z"]
            events.append("body")
            raise RuntimeError("async body failure")
    except RuntimeError:
        events.append("caught")
    return events

async_events = await async_resource_demo()
print(async_events)
assert async_events == [
    "aenter:X", "aenter:Y", "aenter:Z", "body",
    "aexit:Z", "aexit:Y", "aexit:X", "caught",
]

['aenter:X', 'aenter:Y', 'aenter:Z', 'body', 'aexit:Z', 'aexit:Y', 'aexit:X', 'caught']


---
## Problem 16 — Async timeout scope with cancellation-safe cleanup

Create `timeout_scope(seconds)` using `asyncio.timeout` when available.

The wrapper should:

- validate a positive timeout;
- yield control to the body;
- translate `TimeoutError` into a domain-specific `OperationTimedOut` exception;
- preserve the original exception as the cause;
- not translate unrelated errors.

### Solution

In [32]:
class OperationTimedOut(RuntimeError):
    pass


@contextlib.asynccontextmanager
async def timeout_scope(seconds: float) -> AsyncIterator[None]:
    if seconds <= 0:
        raise ValueError("seconds must be positive")
    try:
        async with asyncio.timeout(seconds):
            yield
    except TimeoutError as exc:
        raise OperationTimedOut(f"operation exceeded {seconds} seconds") from exc

In [33]:
async def timeout_demo() -> str:
    try:
        async with timeout_scope(0.01):
            await asyncio.sleep(0.05)
    except OperationTimedOut as exc:
        assert isinstance(exc.__cause__, TimeoutError)
        return str(exc)
    raise AssertionError("timeout was expected")

message = await timeout_demo()
print(message)

operation exceeded 0.01 seconds


---
## Problem 17 — Two-phase "commit or rollback" context manager

Design a generic transaction object with an explicit `.commit()` method.

Semantics:

- Entering takes a deep snapshot.
- If the body raises, roll back.
- If the body exits normally but `.commit()` was not called, roll back.
- If `.commit()` was called and the body exits normally, keep changes.
- A transaction object is single-use.

This design is useful when successful execution is not sufficient evidence that state should be persisted.

### Solution

In [34]:
class ExplicitTransaction:
    def __init__(self, target: MutableMapping[Any, Any]):
        self.target = target
        self._snapshot: dict[Any, Any] | None = None
        self._committed = False
        self._used = False

    def __enter__(self) -> "ExplicitTransaction":
        if self._used:
            raise RuntimeError("transactions are single-use")
        self._used = True
        self._snapshot = copy.deepcopy(dict(self.target))
        return self

    def commit(self) -> None:
        if self._snapshot is None:
            raise RuntimeError("transaction is not active")
        self._committed = True

    def _rollback(self) -> None:
        assert self._snapshot is not None
        self.target.clear()
        self.target.update(self._snapshot)

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        try:
            if exc_type is not None or not self._committed:
                self._rollback()
        finally:
            self._snapshot = None
        return False

In [35]:
settings = {"mode": "safe", "retries": 3}

with ExplicitTransaction(settings) as tx:
    settings["mode"] = "fast"
    # no commit -> rollback
assert settings["mode"] == "safe"

with ExplicitTransaction(settings) as tx:
    settings["mode"] = "fast"
    tx.commit()
assert settings["mode"] == "fast"

try:
    with ExplicitTransaction(settings) as tx:
        settings["retries"] = 0
        tx.commit()
        raise RuntimeError("failure overrides commit")
except RuntimeError:
    pass
assert settings["retries"] == 3
print(settings)

{'mode': 'fast', 'retries': 3}


---
# Capstone — A composable processing session

Build `processing_session` by combining previously solved ideas.

A session should dynamically manage:

- a temporary workspace;
- temporary environment variables;
- optional output capture;
- a timer;
- any number of opened input files;
- an audit dictionary populated even when the body fails.

Use `ExitStack`; do not manually call every `__enter__`/`__exit__` method.

### Solution

In [36]:
@dataclass
class ProcessingSession:
    workspace: Path
    files: list[io.TextIOBase]
    timer: Timer
    captured: CapturedOutput | None
    audit: dict[str, Any]


@contextlib.contextmanager
def processing_session(
    input_paths: Iterable[os.PathLike[str] | str],
    *,
    env: Mapping[str, Any] | None = None,
    capture: bool = True,
) -> Iterator[ProcessingSession]:
    audit: dict[str, Any] = {"ok": False, "exception": None}

    with contextlib.ExitStack() as stack:
        workspace = stack.enter_context(temporary_workspace(prefix="processing-"))
        if env:
            stack.enter_context(temporary_environ(**dict(env)))

        captured = stack.enter_context(capture_output()) if capture else None
        timer = stack.enter_context(Timer())
        files = [
            stack.enter_context(open(path, "r", encoding="utf-8"))
            for path in input_paths
        ]

        session = ProcessingSession(
            workspace=workspace,
            files=files,
            timer=timer,
            captured=captured,
            audit=audit,
        )

        try:
            yield session
        except BaseException as exc:
            audit["exception"] = type(exc).__name__
            raise
        else:
            audit["ok"] = True
        finally:
            # Timer has not exited yet, so record elapsed after the ExitStack closes below.
            audit["workspace"] = str(workspace)

    audit["elapsed"] = timer.elapsed
    audit["files_closed"] = all(file.closed for file in files)

In [37]:
with tempfile.TemporaryDirectory() as source_dir:
    source_root = Path(source_dir)
    source_paths = []
    for index, text in enumerate(["alpha\n", "beta\n", "gamma\n"]):
        path = source_root / f"input-{index}.txt"
        path.write_text(text, encoding="utf-8")
        source_paths.append(path)

    outer_cwd = Path.cwd()
    audit_ref: dict[str, Any] | None = None
    captured_ref: CapturedOutput | None = None

    with processing_session(
        source_paths,
        env={"PROCESSING_MODE": "test"},
        capture=True,
    ) as session:
        audit_ref = session.audit
        captured_ref = session.captured
        combined = "".join(file.read() for file in session.files)
        Path("combined.txt").write_text(combined, encoding="utf-8")
        print("processed", len(session.files), "files")
        assert os.environ["PROCESSING_MODE"] == "test"
        assert Path.cwd() == session.workspace
        assert combined == "alpha\nbeta\ngamma\n"

    assert Path.cwd() == outer_cwd
    assert "PROCESSING_MODE" not in os.environ
    assert audit_ref is not None
    assert audit_ref["ok"] is True
    assert audit_ref["files_closed"] is True
    assert audit_ref["elapsed"] >= 0
    assert captured_ref is not None
    assert captured_ref.stdout.getvalue() == "processed 3 files\n"
    print("audit:", audit_ref)

audit: {'ok': True, 'exception': None, 'workspace': 'C:\\Users\\user1\\AppData\\Local\\Temp\\processing-uh1rba07', 'elapsed': 0.032976099755614996, 'files_closed': True}


---
# Bonus debugging problems

Each snippet contains a context-manager bug. Predict the failure, then compare with the corrected implementation.

## Debugging A — State captured too early

### Bug

```python
class BadPrecision:
    def __init__(self, p):
        self.old = decimal.getcontext().prec  # too early
        self.p = p
```

If global precision changes after construction but before entry, exit restores stale state.

### Fix

Capture state in `__enter__`, or delegate to `decimal.localcontext()`.

In [38]:
class FixedPrecision:
    def __init__(self, precision: int):
        self.precision = precision
        self._old: int | None = None

    def __enter__(self):
        self._old = decimal.getcontext().prec
        decimal.getcontext().prec = self.precision
        return decimal.getcontext()

    def __exit__(self, exc_type, exc_value, traceback):
        assert self._old is not None
        decimal.getcontext().prec = self._old
        return False

## Debugging B — Accidentally swallowing every exception

### Bug

```python
def __exit__(self, exc_type, exc, tb):
    cleanup()
    return True
```

Returning `True` suppresses the active exception. Most cleanup managers should return `False` or `None`.

In [39]:
class CleanupOnly:
    def __init__(self, events: list[str]):
        self.events = events

    def __enter__(self):
        self.events.append("enter")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.events.append("cleanup")
        return False


events = []
try:
    with CleanupOnly(events):
        raise ValueError("visible")
except ValueError:
    events.append("caught")

assert events == ["enter", "cleanup", "caught"]
print(events)

['enter', 'cleanup', 'caught']


## Debugging C — Cleanup that can hide the body exception

If cleanup raises while another exception is already active, the cleanup error becomes the visible exception and the body error appears only in exception chaining. Decide whether that is acceptable.

A common policy is to log cleanup failures and preserve the original exception when one already exists.

In [40]:
class PreserveBodyError:
    def __init__(self, cleanup: Callable[[], None], errors: list[BaseException]):
        self.cleanup = cleanup
        self.errors = errors

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        try:
            self.cleanup()
        except BaseException as cleanup_error:
            if exc_type is None:
                raise
            self.errors.append(cleanup_error)
        return False


def failing_cleanup():
    raise OSError("cleanup failed")

cleanup_errors: list[BaseException] = []
try:
    with PreserveBodyError(failing_cleanup, cleanup_errors):
        raise ValueError("body failed")
except ValueError as exc:
    assert str(exc) == "body failed"

assert isinstance(cleanup_errors[0], OSError)
print("preserved body error; recorded:", cleanup_errors[0])

preserved body error; recorded: cleanup failed


---
# Additional advanced solved problems

The following problems extend the same lifecycle principles to logging, databases, random state, semaphores, tracing, warnings, and plugin systems.

## Problem 18 — Temporary logging level

Implement `temporary_logging_level(logger, level)` so that the logger's exact previous level is restored after normal completion or failure.

In [41]:
import logging


@contextlib.contextmanager
def temporary_logging_level(
    logger: logging.Logger, level: int | str
) -> Iterator[logging.Logger]:
    previous = logger.level
    logger.setLevel(level)
    try:
        yield logger
    finally:
        logger.setLevel(previous)

In [42]:
logger = logging.getLogger("context-manager-lab")
logger.setLevel(logging.WARNING)

with temporary_logging_level(logger, logging.DEBUG):
    assert logger.level == logging.DEBUG

assert logger.level == logging.WARNING

try:
    with temporary_logging_level(logger, logging.ERROR):
        assert logger.level == logging.ERROR
        raise RuntimeError("test")
except RuntimeError:
    pass

assert logger.level == logging.WARNING
print("logging level restored:", logging.getLevelName(logger.level))

logging level restored: WARNING


## Problem 19 — SQLite transaction manager

Implement `sqlite_transaction(connection)` with these rules:

- commit after successful completion;
- roll back after an exception;
- do not suppress the body exception;
- reject use when a transaction is already active, making nesting policy explicit.

In [43]:
import sqlite3


@contextlib.contextmanager
def sqlite_transaction(connection: sqlite3.Connection) -> Iterator[sqlite3.Connection]:
    if connection.in_transaction:
        raise RuntimeError("nested transactions require savepoints and are not supported here")
    try:
        connection.execute("BEGIN")
        yield connection
    except BaseException:
        connection.rollback()
        raise
    else:
        connection.commit()

In [44]:
connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE items (name TEXT PRIMARY KEY, quantity INTEGER)")

with sqlite_transaction(connection) as tx:
    tx.execute("INSERT INTO items VALUES (?, ?)", ("book", 2))

assert connection.execute("SELECT quantity FROM items WHERE name='book'").fetchone() == (2,)

try:
    with sqlite_transaction(connection) as tx:
        tx.execute("INSERT INTO items VALUES (?, ?)", ("pen", 10))
        raise RuntimeError("cancel transaction")
except RuntimeError:
    pass

assert connection.execute("SELECT * FROM items WHERE name='pen'").fetchone() is None
connection.close()
print("SQLite commit and rollback: OK")

SQLite commit and rollback: OK


## Problem 20 — Temporary random seed without disturbing callers

Implement `temporary_random_seed(seed)` using the module-level `random` generator.

The complete RNG state—not only the seed—must be restored after exit.

In [45]:
import random


@contextlib.contextmanager
def temporary_random_seed(seed: Any) -> Iterator[None]:
    previous_state = random.getstate()
    random.seed(seed)
    try:
        yield
    finally:
        random.setstate(previous_state)

In [46]:
random.seed(2026)
first = random.random()
expected_next = random.random()

random.seed(2026)
assert random.random() == first

with temporary_random_seed(7):
    temporary_values = [random.random() for _ in range(3)]

actual_next = random.random()
assert actual_next == expected_next
print("temporary values:", temporary_values)
print("outer stream restored:", actual_next)

temporary values: [0.32383276483316237, 0.15084917392450192, 0.6509344730398537]
outer stream restored: 0.5025157552312506


## Problem 21 — Timed bounded-semaphore slot

Implement `bounded_semaphore_slot(semaphore, timeout)`.

- Raise `SlotUnavailable` if acquisition times out.
- Release only if acquisition succeeded.
- Always release after a body exception.

In [47]:
class SlotUnavailable(TimeoutError):
    pass


@contextlib.contextmanager
def bounded_semaphore_slot(
    semaphore: threading.BoundedSemaphore,
    timeout: float | None = None,
) -> Iterator[None]:
    acquired = semaphore.acquire(timeout=timeout)
    if not acquired:
        raise SlotUnavailable(f"no semaphore slot became available within {timeout} seconds")
    try:
        yield
    finally:
        semaphore.release()

In [48]:
semaphore = threading.BoundedSemaphore(1)

with bounded_semaphore_slot(semaphore, timeout=0.01):
    try:
        with bounded_semaphore_slot(semaphore, timeout=0.01):
            raise AssertionError("second acquisition should not succeed")
    except SlotUnavailable:
        pass

# The outer slot was released.
assert semaphore.acquire(blocking=False)
semaphore.release()

try:
    with bounded_semaphore_slot(semaphore, timeout=0.01):
        raise ValueError("body failure")
except ValueError:
    pass

assert semaphore.acquire(blocking=False)
semaphore.release()
print("semaphore lifecycle: OK")

semaphore lifecycle: OK


## Problem 22 — Re-entrant operation tracing tree

Create a tracer whose `span(name)` context manager builds a tree of nested operations.

Requirements:

- nesting must be re-entrant;
- each node records elapsed time and failure type;
- parent selection should be context-local;
- completed top-level spans are retained as roots.

In [49]:
@dataclass
class TraceNode:
    name: str
    children: list["TraceNode"] = field(default_factory=list)
    elapsed: float = 0.0
    exception: str | None = None


class Tracer:
    def __init__(self):
        self.roots: list[TraceNode] = []
        self._current = contextvars.ContextVar(f"trace_current_{id(self)}", default=None)

    @contextlib.contextmanager
    def span(self, name: str) -> Iterator[TraceNode]:
        node = TraceNode(name=name)
        parent = self._current.get()
        if parent is None:
            self.roots.append(node)
        else:
            parent.children.append(node)

        token = self._current.set(node)
        started = perf_counter()
        try:
            yield node
        except BaseException as exc:
            node.exception = type(exc).__name__
            raise
        finally:
            node.elapsed = perf_counter() - started
            self._current.reset(token)

In [50]:
tracer = Tracer()
with tracer.span("request"):
    with tracer.span("parse"):
        sum(range(100))
    try:
        with tracer.span("database"):
            raise ConnectionError("offline")
    except ConnectionError:
        pass

root = tracer.roots[0]
assert root.name == "request"
assert [child.name for child in root.children] == ["parse", "database"]
assert root.children[1].exception == "ConnectionError"
assert all(node.elapsed >= 0 for node in [root, *root.children])
print(root)

TraceNode(name='request', children=[TraceNode(name='parse', children=[], elapsed=5.2996911108493805e-06, exception=None), TraceNode(name='database', children=[], elapsed=9.200070053339005e-06, exception='ConnectionError')], elapsed=4.2300205677747726e-05, exception=None)


## Problem 23 — Async semaphore permit with timeout

Implement `async_semaphore_slot(semaphore, timeout=None)`.

Use `asyncio.wait_for` when a timeout is supplied, translate timeout into `SlotUnavailable`, and release only after successful acquisition.

In [51]:
@contextlib.asynccontextmanager
async def async_semaphore_slot(
    semaphore: asyncio.Semaphore,
    timeout: float | None = None,
) -> AsyncIterator[None]:
    try:
        if timeout is None:
            await semaphore.acquire()
        else:
            await asyncio.wait_for(semaphore.acquire(), timeout=timeout)
    except TimeoutError as exc:
        raise SlotUnavailable(f"no async slot available within {timeout} seconds") from exc

    try:
        yield
    finally:
        semaphore.release()

In [52]:
async def async_semaphore_demo() -> None:
    semaphore = asyncio.Semaphore(1)
    async with async_semaphore_slot(semaphore, timeout=0.1):
        try:
            async with async_semaphore_slot(semaphore, timeout=0.01):
                raise AssertionError("second permit should not be available")
        except SlotUnavailable:
            pass

    assert semaphore._value == 1  # controlled demonstration check

await async_semaphore_demo()
print("async semaphore lifecycle: OK")

async semaphore lifecycle: OK


## Problem 24 — Temporary warning policy

Implement `temporary_warning_filter(action, *, category=Warning)`.

Use `warnings.catch_warnings()` so the process-wide filter list is restored exactly after exit.

In [53]:
import warnings


@contextlib.contextmanager
def temporary_warning_filter(
    action: str,
    *,
    category: type[Warning] = Warning,
) -> Iterator[None]:
    with warnings.catch_warnings():
        warnings.simplefilter(action, category)
        yield

In [54]:
with temporary_warning_filter("error", category=UserWarning):
    try:
        warnings.warn("promoted", UserWarning)
    except UserWarning as exc:
        assert str(exc) == "promoted"
    else:
        raise AssertionError("warning should have become an exception")

# Outside the context, capture the warning normally instead of raising it.
with warnings.catch_warnings(record=True) as recorded:
    warnings.simplefilter("always")
    warnings.warn("ordinary", UserWarning)

assert len(recorded) == 1
print("warning policy restored:", recorded[0].message)

warning policy restored: ordinary


## Problem 25 — Dynamic plugin session

A plugin may return a context manager from `activate()`, or `None` if no lifecycle management is required.

Implement `plugin_session(plugins)` with `ExitStack` so that:

- plugins activate in input order;
- managed plugins deactivate in reverse order;
- partially activated sessions clean up correctly;
- the body receives the original plugin objects.

In [55]:
class DemoPlugin:
    def __init__(
        self,
        name: str,
        events: list[str],
        *,
        managed: bool = True,
        fail: bool = False,
    ):
        self.name = name
        self.events = events
        self.managed = managed
        self.fail = fail

    def activate(self):
        plugin = self
        if not self.managed:
            self.events.append(f"activate:{self.name}")
            if self.fail:
                raise RuntimeError(f"plugin failed: {self.name}")
            return None

        @contextlib.contextmanager
        def lifecycle():
            plugin.events.append(f"activate:{plugin.name}")
            if plugin.fail:
                raise RuntimeError(f"plugin failed: {plugin.name}")
            try:
                yield plugin
            finally:
                plugin.events.append(f"deactivate:{plugin.name}")

        return lifecycle()


@contextlib.contextmanager
def plugin_session(plugins: Iterable[DemoPlugin]) -> Iterator[list[DemoPlugin]]:
    plugin_list = list(plugins)
    with contextlib.ExitStack() as stack:
        for plugin in plugin_list:
            manager = plugin.activate()
            if manager is not None:
                stack.enter_context(manager)
        yield plugin_list

In [56]:
plugin_events: list[str] = []
plugins = [
    DemoPlugin("A", plugin_events),
    DemoPlugin("B", plugin_events, managed=False),
    DemoPlugin("C", plugin_events),
]

with plugin_session(plugins) as active:
    assert active == plugins
    plugin_events.append("body")

assert plugin_events == [
    "activate:A", "activate:B", "activate:C", "body",
    "deactivate:C", "deactivate:A",
]
print(plugin_events)

partial_events: list[str] = []
try:
    with plugin_session([
        DemoPlugin("A", partial_events),
        DemoPlugin("B", partial_events, fail=True),
    ]):
        pass
except RuntimeError:
    pass

assert partial_events == ["activate:A", "activate:B", "deactivate:A"]
print("partial activation cleanup:", partial_events)

['activate:A', 'activate:B', 'activate:C', 'body', 'deactivate:C', 'deactivate:A']
partial activation cleanup: ['activate:A', 'activate:B', 'deactivate:A']


# Summary

The most important invariant is:

> **After the context exits, externally visible state should be correct regardless of how the body exited.**

The body may return normally, raise an ordinary exception, be cancelled asynchronously, or fail after only some resources were acquired. Correct context managers make every one of those paths explicit and testable.